In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!git clone https://github.com/RGenDiff/hgvt.git

Cloning into 'hgvt'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 37 (delta 6), reused 37 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 872.07 KiB | 32.30 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [2]:
%cd hgvt

/kaggle/working/hgvt


In [3]:
!pip install -e .

Obtaining file:///kaggle/working/hgvt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hgvt (pyproject.toml) ... done
  Created wheel for hgvt: filename=hgvt-0.0.1-0.editable-py3-none-any.whl size=8119 sha256=57d2f7b79f832dbf26491d6633373d181f6c1ec628e80a631da9c10ca5675b3c
  Stored in directory: /tmp/pip-ephem-wheel-cache-svi8462p/wheels/6b/51/0e/451b5fca449bd04de3cf447b7197a9f9c27db46ad93ab7ba9e
Successfully built hgvt


In [4]:
import torch
print(torch.__version__, torch.cuda.is_available())

2.10.0+cu128 True


In [5]:
# unzip your upload (or use the git clone you already did)
!unzip -q hgvt-main.zip && cd hgvt-main

# core package
!pip install -e .

# additional deps actually imported by the code but NOT in pyproject.toml:
!pip install pytorch-lightning scipy kornia tqdm packaging

# only needed if you want the ImageNet/WebDataset path (not CIFAR):
!pip install webdataset

unzip:  cannot find or open hgvt-main.zip, hgvt-main.zip.zip or hgvt-main.zip.ZIP.
Obtaining file:///kaggle/working/hgvt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hgvt (pyproject.toml) ... done
  Created wheel for hgvt: filename=hgvt-0.0.1-0.editable-py3-none-any.whl size=8119 sha256=f31cc476facf8fc9f3d8acdd7204dcb4612458960204b7ba326164307e62c8ea
  Stored in directory: /tmp/pip-ephem-wheel-cache-fuacqvkm/wheels/6b/51/0e/451b5fca449bd04de3cf447b7197a9f9c27db46ad93ab7ba9e
Successfully built hgvt
  Attempting uninstall: hgvt
    Found existing installation: hgvt 0.0.1
    Uninstalling hgvt-0.0.1:
      Successfully uninstalled hgvt-0.0.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 5.6 MB/s eta 0:00:00


In [6]:
!pip install -e .
!pip install pytorch-lightning scipy kornia tqdm packaging webdataset

Obtaining file:///kaggle/working/hgvt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hgvt (pyproject.toml) ... done
  Created wheel for hgvt: filename=hgvt-0.0.1-0.editable-py3-none-any.whl size=8119 sha256=677dc8b1acabdd8b8e36dd3dc31ca392ec001fab2f788005b29db2778276e468
  Stored in directory: /tmp/pip-ephem-wheel-cache-wf6dz10d/wheels/6b/51/0e/451b5fca449bd04de3cf447b7197a9f9c27db46ad93ab7ba9e
Successfully built hgvt
  Attempting uninstall: hgvt
    Found existing installation: hgvt 0.0.1
    Uninstalling hgvt-0.0.1:
      Successfully uninstalled hgvt-0.0.1


In [7]:
!pip install webdataset kornia

In [8]:
%%writefile /kaggle/working/hgvt/train.py
"""
train.py — CLI entry point for HgVT training.
"""

import argparse
import os
import time
import contextlib
import io

from omegaconf import OmegaConf
import torch
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor

from hgvt.util import instantiate_from_config


def get_parser():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", type=str, required=True)
    parser.add_argument("--name", type=str, default=None)
    parser.add_argument("--logdir", type=str, default="logs")
    parser.add_argument("--gpus", type=int, default=1)
    parser.add_argument("--precision", type=str, default="32")
    parser.add_argument("--resume", type=str, default=None)
    parser.add_argument("--seed", type=int, default=21)
    parser.add_argument("--max_steps", type=int, default=None,
        help="Overrides the config's max_steps if given. If omitted, falls back to the yaml's own lightning.trainer.max_steps.")
    parser.add_argument("--limit_train_batches", type=str, default=None)
    parser.add_argument("--limit_val_batches", type=str, default=None)
    return parser


def normalize_precision(precision_arg):
    mapping = {"32": "32", "16": "16", "bf16": "bf16"}
    return mapping.get(precision_arg, precision_arg)


def parse_int_or_float(value):
    if value is None:
        return None
    try:
        return int(value)
    except ValueError:
        return float(value)


def build_trainer(args, trainer_extra_cfg, logger, callbacks):
    use_gpu = args.gpus > 0 and torch.cuda.is_available()
    if args.gpus > 0 and not torch.cuda.is_available():
        print("WARNING: --gpus > 0 requested but no CUDA device is visible. Falling back to CPU.")

    accelerator = "gpu" if use_gpu else "cpu"
    devices = args.gpus if use_gpu else 1

    trainer_kwargs = dict(trainer_extra_cfg)
    for key in ("gpus", "strategy", "accelerator", "devices", "precision",
                "max_steps", "limit_train_batches", "limit_val_batches"):
        trainer_kwargs.pop(key, None)

    # CLI flag overrides the yaml if given; otherwise fall back to the
    # yaml's own value instead of silently dropping it (this was the bug:
    # previously, if you didn't pass --max_steps, the yaml's max_steps got
    # stripped above and never restored, so PL defaulted to max_epochs=1000).
    if args.max_steps is not None:
        trainer_kwargs["max_steps"] = args.max_steps
    elif "max_steps" in trainer_extra_cfg:
        trainer_kwargs["max_steps"] = trainer_extra_cfg["max_steps"]

    if args.limit_train_batches is not None:
        trainer_kwargs["limit_train_batches"] = parse_int_or_float(args.limit_train_batches)
    elif "limit_train_batches" in trainer_extra_cfg:
        trainer_kwargs["limit_train_batches"] = trainer_extra_cfg["limit_train_batches"]

    if args.limit_val_batches is not None:
        trainer_kwargs["limit_val_batches"] = parse_int_or_float(args.limit_val_batches)
    elif "limit_val_batches" in trainer_extra_cfg:
        trainer_kwargs["limit_val_batches"] = trainer_extra_cfg["limit_val_batches"]

    if devices > 1:
        trainer_kwargs["strategy"] = "ddp"

    precision = normalize_precision(args.precision)

    try:
        trainer = pl.Trainer(
            accelerator=accelerator, devices=devices, precision=precision,
            logger=logger, callbacks=callbacks, **trainer_kwargs,
        )
    except Exception as e:
        if precision in ("16", "bf16"):
            print(f"Trainer construction failed with precision='{precision}' ({e}); retrying with '{precision}-mixed'.")
            trainer = pl.Trainer(
                accelerator=accelerator, devices=devices, precision=f"{precision}-mixed",
                logger=logger, callbacks=callbacks, **trainer_kwargs,
            )
        else:
            raise
    return trainer


def main():
    parser = get_parser()
    args = parser.parse_args()

    pl.seed_everything(args.seed)

    config = OmegaConf.load(args.config)
    model_config = config.model
    data_config = config.data
    lightning_config = config.get("lightning", OmegaConf.create())
    trainer_config = lightning_config.get("trainer", OmegaConf.create())
    ckpt_config = lightning_config.get("modelcheckpoint", OmegaConf.create()).get("params", OmegaConf.create())

    run_name = args.name or os.path.splitext(os.path.basename(args.config))[0]
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    run_dir = os.path.join(args.logdir, f"{timestamp}_{run_name}")
    ckpt_dir = os.path.join(run_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)

    print(f"Run directory: {run_dir}")

    # --- build model (stdout suppressed: silences the ~50 lines of
    # per-layer construction prints from hgvt/network/*.py) ---
    with contextlib.redirect_stdout(io.StringIO()):
        model = instantiate_from_config(model_config)
    if args.resume is not None:
        print(f"Resuming weights from {args.resume}")

    # --- build data module ---
    datamodule = instantiate_from_config(data_config)

    inner_factory = getattr(datamodule, "dataset_factory", None)
    if inner_factory is not None and hasattr(inner_factory, "prepare_data"):
        print("Preparing/downloading dataset via inner data factory...")
        inner_factory.prepare_data()

    logger = TensorBoardLogger(save_dir=args.logdir, name=run_name, version=timestamp)

    checkpoint_kwargs = dict(ckpt_config)
    checkpoint_kwargs.setdefault("dirpath", ckpt_dir)
    checkpoint_kwargs.setdefault("filename", "{epoch}-{step}")
    monitor = model_config.params.get("monitor", None)
    if monitor is not None:
        checkpoint_kwargs.setdefault("monitor", monitor)
        checkpoint_kwargs.setdefault("mode", model_config.params.get("mon_mode", "min"))
    checkpoint_kwargs.setdefault("save_top_k", model_config.params.get("save_top_k", 1))
    checkpoint_callback = ModelCheckpoint(**checkpoint_kwargs)

    lr_monitor = LearningRateMonitor(logging_interval="step")
    callbacks = [checkpoint_callback, lr_monitor]

    trainer = build_trainer(args, trainer_config, logger, callbacks)
    trainer.fit(model, datamodule=datamodule, ckpt_path=args.resume)


if __name__ == "__main__":
    main()

Writing /kaggle/working/hgvt/train.py


In [9]:
"""
Consolidated patch for hgvt/hypergraph.py — applies all 4 fixes in one
shot on a freshly cloned repo:
  1. rank_zero_only import path (PL 2.0 moved it)
  2. stop_validation() distributed guard (works without torch.distributed init)
  3. test_epoch_end -> on_test_epoch_end (removed hook name in PL 2.0)
  4. val/ce_loss, val/top1, val/top5 now show in the live progress bar
"""

path = "/kaggle/working/hgvt/hgvt/hypergraph.py"
with open(path) as f:
    content = f.read()

# Fix 1: rank_zero_only import
old1 = "from pytorch_lightning.utilities.distributed import rank_zero_only"
new1 = "from pytorch_lightning.utilities.rank_zero import rank_zero_only"
assert old1 in content, "FIX1 pattern not found"
content = content.replace(old1, new1, 1)

# Fix 2: stop_validation distributed guard
old2 = """    def stop_validation(self, reduce_time=False):

        # get the world size
        world_size = dist.get_world_size()
        
        # Concatenate features from all batches
        losses_m = torch.tensor([self.losses_m.avg], device=self.device)
        top1_m = torch.tensor([self.top1_m.avg], device=self.device)
        top5_m = torch.tensor([self.top5_m.avg], device=self.device)
        kentropy_m = torch.tensor([self.k_entropy_m.avg], device=self.device)
        kdiversity_m = torch.tensor([self.k_diversity_m.avg], device=self.device)
            
        # allocate the gather features from all GPUs
        gathered_losses_m = [torch.zeros_like(losses_m) for _ in range(world_size)]
        gathered_top1_m = [torch.zeros_like(top1_m) for _ in range(world_size)]
        gathered_top5_m = [torch.zeros_like(top5_m) for _ in range(world_size)]
        gathered_kentropy_m = [torch.zeros_like(kentropy_m) for _ in range(world_size)]
        gathered_kdiversity_m = [torch.zeros_like(kdiversity_m) for _ in range(world_size)]
        
        # perform the gather
        dist.all_gather(gathered_losses_m, losses_m)
        dist.all_gather(gathered_top1_m, top1_m)
        dist.all_gather(gathered_top5_m, top5_m)
        dist.all_gather(gathered_kentropy_m, kentropy_m)
        dist.all_gather(gathered_kdiversity_m, kdiversity_m)
        
        # Concatenate features from all GPUs and mean
        final_losses_m = torch.cat(gathered_losses_m, dim=0).mean()
        final_top1_m = torch.cat(gathered_top1_m, dim=0).mean()
        final_top5_m = torch.cat(gathered_top5_m, dim=0).mean()
        final_kentropy_m = torch.cat(gathered_kentropy_m, dim=0).mean()
        final_kdiversity_m = torch.cat(gathered_kdiversity_m, dim=0).mean()

        if reduce_time:
            batch_time_m = torch.tensor([self.batch_time_m.avg], device=self.device)
            gathered_batch_time_m = [torch.zeros_like(batch_time_m) for _ in range(world_size)]
            dist.all_gather(gathered_batch_time_m, batch_time_m)
            final_batch_time_m = torch.cat(gathered_batch_time_m, dim=0).mean()
        else:
            final_batch_time_m = None

        return final_losses_m, final_top1_m, final_top5_m, final_kentropy_m, final_kdiversity_m, final_batch_time_m"""
new2 = """    def stop_validation(self, reduce_time=False):

        # only gather across processes if torch.distributed is actually
        # initialized (single-GPU/non-DDP runs never initialize it)
        distributed = dist.is_available() and dist.is_initialized()
        world_size = dist.get_world_size() if distributed else 1

        # Concatenate features from all batches
        losses_m = torch.tensor([self.losses_m.avg], device=self.device)
        top1_m = torch.tensor([self.top1_m.avg], device=self.device)
        top5_m = torch.tensor([self.top5_m.avg], device=self.device)
        kentropy_m = torch.tensor([self.k_entropy_m.avg], device=self.device)
        kdiversity_m = torch.tensor([self.k_diversity_m.avg], device=self.device)

        if distributed:
            gathered_losses_m = [torch.zeros_like(losses_m) for _ in range(world_size)]
            gathered_top1_m = [torch.zeros_like(top1_m) for _ in range(world_size)]
            gathered_top5_m = [torch.zeros_like(top5_m) for _ in range(world_size)]
            gathered_kentropy_m = [torch.zeros_like(kentropy_m) for _ in range(world_size)]
            gathered_kdiversity_m = [torch.zeros_like(kdiversity_m) for _ in range(world_size)]

            dist.all_gather(gathered_losses_m, losses_m)
            dist.all_gather(gathered_top1_m, top1_m)
            dist.all_gather(gathered_top5_m, top5_m)
            dist.all_gather(gathered_kentropy_m, kentropy_m)
            dist.all_gather(gathered_kdiversity_m, kdiversity_m)

            final_losses_m = torch.cat(gathered_losses_m, dim=0).mean()
            final_top1_m = torch.cat(gathered_top1_m, dim=0).mean()
            final_top5_m = torch.cat(gathered_top5_m, dim=0).mean()
            final_kentropy_m = torch.cat(gathered_kentropy_m, dim=0).mean()
            final_kdiversity_m = torch.cat(gathered_kdiversity_m, dim=0).mean()
        else:
            final_losses_m = losses_m.mean()
            final_top1_m = top1_m.mean()
            final_top5_m = top5_m.mean()
            final_kentropy_m = kentropy_m.mean()
            final_kdiversity_m = kdiversity_m.mean()

        if reduce_time:
            batch_time_m = torch.tensor([self.batch_time_m.avg], device=self.device)
            if distributed:
                gathered_batch_time_m = [torch.zeros_like(batch_time_m) for _ in range(world_size)]
                dist.all_gather(gathered_batch_time_m, batch_time_m)
                final_batch_time_m = torch.cat(gathered_batch_time_m, dim=0).mean()
            else:
                final_batch_time_m = batch_time_m.mean()
        else:
            final_batch_time_m = None

        return final_losses_m, final_top1_m, final_top5_m, final_kentropy_m, final_kdiversity_m, final_batch_time_m"""
assert old2 in content, "FIX2 pattern not found"
content = content.replace(old2, new2, 1)

# Fix 3: test_epoch_end -> on_test_epoch_end
old3 = "    def test_epoch_end(self, *args, **kwargs):"
new3 = "    def on_test_epoch_end(self, *args, **kwargs):"
assert old3 in content, "FIX3 pattern not found"
content = content.replace(old3, new3, 1)

# Fix 4: val metrics show in live progress bar
old4 = """        self.log('val/ce_loss', final_losses_m,  prog_bar=False, logger=True, on_step=False, on_epoch=True)     
        self.log('val/top1', final_top1_m,  prog_bar=False, logger=True, on_step=False, on_epoch=True)  
        self.log('val/top5', final_top5_m,  prog_bar=False, logger=True, on_step=False, on_epoch=True) """
new4 = """        self.log('val/ce_loss', final_losses_m,  prog_bar=True, logger=True, on_step=False, on_epoch=True)     
        self.log('val/top1', final_top1_m,  prog_bar=True, logger=True, on_step=False, on_epoch=True)  
        self.log('val/top5', final_top5_m,  prog_bar=True, logger=True, on_step=False, on_epoch=True) """
assert old4 in content, "FIX4 pattern not found"
content = content.replace(old4, new4, 1)

with open(path, "w") as f:
    f.write(content)

print("all 4 hypergraph.py fixes applied: rank_zero_only import, stop_validation guard, on_test_epoch_end rename, val prog_bar")

all 4 hypergraph.py fixes applied: rank_zero_only import, stop_validation guard, on_test_epoch_end rename, val prog_bar


In [10]:
%%writefile /kaggle/working/hgvt/configs/hgvt_mu_smoketest.yaml
model:
  base_learning_rate: 1.0 # LR scheduler multiplier
  target: hgvt.hypergraph.HyperGraph
  params:
    monitor: "val/ce_loss"
    save_top_k: 1
    use_ema: False
    ema_decay: 0.99996
    log_finegrained: False
    compile_model: False
    log_experts: True
    # === Criteron args ===
    label_smoothing: 0.1   # cross-entropy label smoothing
    mixup_active: True     # mixup is active when training, so we need to change the CE method
    # === loss weights ===
    diversity_weight: 1.0 
    pop_weight: 1.0
    exp_weight: 1.0
    # === optimizer parameters ===
    opt_beta1: 0.9
    opt_beta2: 0.999
    opt_weight_decay: 0.05
    opt_eps: 1.0e-8
    # Note: for CIFAR-100, each epoch is 98 steps @ batch=512

    # note with DINOv2
    # patch_embed_lr_mult: 0.2
    timm_optimizer: # opt for timm
        opt: adamw
        #opt: fusedadamw  # requires Apex built with CUDA extensions; switch back once Apex is confirmed working
        lr: 1e-3
        opt_eps: 1e-8
        weight_decay: .05
        opt_betas: [0.9, 0.999]
        momentum: 0.9

    # note have to divide numbers by 2 for 2 repeats
    scheduler_config: # for timm
        sched: cosine
        #epochs: 75 # 150
        epochs: 100 # 100 (reduced for faster convergence on subset)
        #decay_epochs: 30 # not used
        warmup_epochs: 3 # reduced for faster convergence on subset
        cooldown_epochs: 5 # 10
        #patience_epochs: 10 # not used
        #decay_rate: 0.1 # not used
        warmup_lr: 1e-6
        min_lr: 1e-5
        lr_noise: Null
        lr_noise_pct: 0.67
        lr_noise_std: 1.0
        seed: 21
        lr_cycle_mul: 1
        lr_cycle_limit: 1
        
    model_config:
        target:  hgvt.network.hypergraph.HypergraphNet
        params:
            # === I/O params ===
            in_features: 3
            image_size: 32
            patch_size: 4   # num_nodes = 64
            num_classes: 100
            # === Arch params ===
            num_layers: 10
            hidden_dim: 64 
            num_heads: 2 # 64/2 = 32
            adj_dim: Null
            hnode_capacity: 5 # hidden nodes to 5
            edge_capacity: 8 # edge capacity to 8
            hedge_capacity: 4 # hidden edges to 4
            # === primary state config ===
            split_mode: True # (v_a,v_f) vs v_a=v_f seems to be important  
            joint_mlp: True # FIX: paper's Mu config uses True (was False)
            bind_qkv: False #Q=K=V
            use_stem: True # uses patch embedding if false
            # === regularization helpers ===                
            dropout: 0.1 # class dropout
            drop_path: 0.1 # stochastic dropout
            drop_decay: False # FIX: paper's Mu config uses False (was True)
            skip_last: True # used to skip last blocks with no useful gradient
            weight_init: True
            virtual_init: False
            train_pe: False
            use_virtual_pemb: True # virtual position embeddings for edges and hidden nodes
            use_diversity_loss: True # FIX: was False — this is the collapse-prevention bug (was disabled despite the comment saying otherwise)
            output_ln: True
            use_xformers: False
            pop_level: [10.66, 0.5, 0.5] # set to 1/6 of all tokens as max, and 0.5 as min, 0.5 sim weight
            # == pooling params ===
            pool_type: 'expert' # expert, max, mean, image
            expert_topk: 1 # only use one expert hyperedge per prediction
            expert_ce_weight: 0.1 # cross-entropy prediction weight
            expert_noise_level: 0.1 # FIX: paper's Mu config uses 0.1 (was 1e-3)
            expert_dropout: 0.1 # dropout on prediction
            expert_label_smoothing: 0.0 # FIX: paper's Mu config uses 0.0 (was 0.1)
            expert_use_gate: False # use FFN gate for prediction

data:
  target: hgvt.data.prefetch_loader.PrefetchLoaderFromConfig
  params:
    use_async: False
    batch_size: 512 # x 1 accum x 1 GPU = 512
    async_config:
        train:
            fp16: True
            # === random erase ===
            re_mode: 'pixel'
            re_prob: 0.25
            re_count: 1
            re_num_splits: 0
            # === mixup args ===
            label_smoothing: 0.1   # cross-entropy label smoothing
            mixup: 0.8             # mixup alpha, mixup enabled if > 0
            cutmix: 1.0            # cutmix alpha, cutmix enabled if > 0
            cutmix_minmax: Null    # cutmix alpha, cutmix enabled if > 0
            mixup_prob: 0.8        # Probability of performing mixup or cutmix when either/both is enabled
            mixup_switch_prob: 0.5 # Probability of switching to cutmix when both mixup and cutmix enabled
            mixup_mode: 'batch'    # How to apply mixup/cutmix params. Per "batch", "pair", or "elem"
            mixup_off_epoch: 0     # Turn off mixup after this epoch, disabled if 0
            num_classes: 100      # number of classes for label smoothing
        val:
            fp16: True
        test:
            fp16: True
            
    factory_config:
        target: hgvt.data.torchvision_data_module.TorchDataModule
        params:
            batch_size: 512
            dataset_name: CIFAR100
            tar_base: "/vault/imagenet/shards/" # not used
            num_workers: 4
            multinode: False
            pin_memory: True
            generate_cls: True
            train:
              shards: 'train/{00000..00127}.tar' # not used
              shuffle: 10000
              image_key: jpg # not used
              repeat: 1
              repeat_aug: 2
              subset_size: 30000
              cls_dict_path: '/vault/imagenet/imagenet_classdict.json' # not used
              transforms:
                  img_size: 32
                  color_jitter: 0.4
                  re_mode: 'pixel'
                  re_prob: 0.25
                  re_count: 1
                  re_split: False
                  scale: [0.08, 1.0]
                  ratio: [0.75, 1.33]
                  hflip: 0.5
                  vflip: 0.0
                  mean: [0.5, 0.5, 0.5]
                  std: [0.5, 0.5, 0.5]
                  num_aug_splits: 0
                  auto_augment: 'rand-m9-mstd0.5-inc1'
                  interpolation: 'random'
                  use_prefetcher: True
                  crop_pct: Null
        
        
            validation:
              shards: 'train/{00128..00128}.tar' # not used 
              shuffle: 0
              image_key: jpg # not used
              cls_dict_path: '/vault/imagenet/imagenet_classdict.json' # not used
              repeats: 1 # webdata loader requires this for 1 image, 0=infinite
              transforms:
                  mean: [0.5, 0.5, 0.5]
                  std: [0.5, 0.5, 0.5]
                  img_size: 32
                  no_aug: True
                  use_prefetcher: True
        
            test:
              shards: 'val1/{00000..00007}.tar' # not used
              shuffle: 0
              image_key: jpg # not used
              cls_dict_path: '/vault/imagenet/val_set_id_mapping.json' # not used
              repeats: 1 #webdata loader requires this for 1 image, 0=infinite
              subset_size: 5000
              transforms: 
                  mean: [0.5, 0.5, 0.5]
                  std: [0.5, 0.5, 0.5]
                  img_size: 32
                  no_aug: True
                  use_prefetcher: True

lightning:
  modelcheckpoint:
    params:
      # subset run: ~500 steps ~= every 4 epochs at 118 steps/epoch (30k images, effective batch 256)
      every_n_train_steps: 500
      save_last: True


  trainer:
    accelerator: gpu
    strategy: ddp_find_unused_parameters_false # all paths are used, so we can do this to be faster
    num_sanity_val_steps: 0
    check_val_every_n_epoch: 1 # check at the end of each training epoch
    max_steps: 11800 # 100 epochs * 118 steps/epoch (30k images, effective batch 256)
    accumulate_grad_batches: 1
    gradient_clip_val: 1.0

Writing /kaggle/working/hgvt/configs/hgvt_mu_smoketest.yaml


In [11]:
!sed -i "s/from pytorch_lightning.utilities.distributed import rank_zero_only/from pytorch_lightning.utilities.rank_zero import rank_zero_only/" /kaggle/working/hgvt/hgvt/hypergraph.py

In [12]:
import os
os.environ["ATTN_PRECISION"] = "fp32"
os.environ["USE_XFORMERS"] = "1"
os.environ["USE_APEX"] = "0"

In [13]:
path = "/kaggle/working/hgvt/hgvt/data/torchvision_data_module.py"
with open(path) as f:
    content = f.read()

# --- patch 1: numpy/tensor fix for use_prefetcher (current timm returns Tensor, not ndarray) ---
old1 = """            if self.use_prefetcher:
                ximg = torch.from_numpy(ximg)"""
new1 = """            if self.use_prefetcher and isinstance(ximg, np.ndarray):
                ximg = torch.from_numpy(ximg)"""
assert old1 in content, "pattern 1 not found — check indentation matches the file"
content = content.replace(old1, new1)

# --- patch 2: add subset_size support for train/test datasets ---
old2 = """        # Create appropriate dataset (train, val, or test)
        if train:
            dataset = dataset_instance.create_train(self.batch_size)
        elif test:
            dataset = dataset_instance.create_test(self.batch_size)
        else:
            dataset = dataset_instance.create_val(self.batch_size)"""
new2 = """        # Create appropriate dataset (train, val, or test)
        if train:
            dataset = dataset_instance.create_train(self.batch_size)
        elif test:
            dataset = dataset_instance.create_test(self.batch_size)
        else:
            dataset = dataset_instance.create_val(self.batch_size)

        # optionally subsample train/test to a fixed size (deterministic, seeded)
        subset_size = dataset_config.get("subset_size", None)
        if (train or test) and subset_size is not None and subset_size < len(dataset):
            g = torch.Generator().manual_seed(self.seed)
            indices = torch.randperm(len(dataset), generator=g)[:subset_size].tolist()
            dataset = torch.utils.data.Subset(dataset, indices)"""
assert old2 in content, "pattern 2 not found — check indentation matches the file"
content = content.replace(old2, new2)

with open(path, "w") as f:
    f.write(content)

print("patched: numpy/tensor fix + subset_size support both applied")

patched: numpy/tensor fix + subset_size support both applied


In [14]:
path = "/kaggle/working/hgvt/hgvt/data/prefetch_loader.py"
with open(path) as f:
    content = f.read()

old = """    @property
    def sampler(self):
        return self.loader.sampler

    @property
    def dataset(self):
        return self.loader.dataset

class AsyncPrefetchLoader(PrefetchLoader):"""

new = """    @property
    def sampler(self):
        return self.loader.sampler

    @property
    def dataset(self):
        return self.loader.dataset

    @property
    def batch_sampler(self):
        return self.loader.batch_sampler

class AsyncPrefetchLoader(PrefetchLoader):"""

assert old in content, "pattern not found — check file matches expected content"
content = content.replace(old, new, 1)

with open(path, "w") as f:
    f.write(content)

print("patched")

patched


In [ ]:
!cd /kaggle/working/hgvt && python train.py \
    --config configs/hgvt_mu_smoketest.yaml \
    --name cifar_subset_baseline_v2 \
    --logdir /kaggle/working/logs \
    --gpus 1 \
    --precision bf16

In [ ]:
!cd /kaggle/working/hgvt && python train.py \
    --config configs/hgvt_mu_smoketest.yaml \
    --name cifar_subset_baseline_v2 \
    --logdir /kaggle/working/logs \
    --gpus 1 \
    --precision bf16 \
    --resume /kaggle/working/logs/20260719-052740_cifar_subset_baseline_v2/checkpoints/last.ckpt

In [ ]:
!ls -dt /kaggle/working/logs/*cifar_subset_baseline_v2*/checkpoints/last.ckpt | head -1

In [15]:
%%writefile /kaggle/working/hgvt/configs/hgvt_mu_phase2.yaml
model:
  base_learning_rate: 1.0
  target: hgvt.hypergraph.HyperGraph
  params:
    ckpt_path: "/kaggle/working/logs/20260719-102603_cifar_subset_baseline_v2/checkpoints/last.ckpt"   # weights-only load, not --resume
    monitor: "val/ce_loss"
    save_top_k: 1
    use_ema: False
    ema_decay: 0.99996
    log_finegrained: False
    compile_model: False
    log_experts: True
    label_smoothing: 0.1
    mixup_active: True
    diversity_weight: 1.0
    pop_weight: 1.0
    exp_weight: 1.0
    opt_beta1: 0.9
    opt_beta2: 0.999
    opt_weight_decay: 0.05
    opt_eps: 1.0e-8

    timm_optimizer:
        opt: adamw
        lr: 3e-4  # lower peak than phase 1's 1e-3 — weights are already trained, don't want a jarring restart
        opt_eps: 1e-8
        weight_decay: .05
        opt_betas: [0.9, 0.999]
        momentum: 0.9

    scheduler_config:
        sched: cosine
        epochs: 100          # a REAL 100-epoch cycle, matched to max_steps below
        warmup_epochs: 2     # small warmup so the 3e-4 restart doesn't jolt trained weights
        cooldown_epochs: 5
        warmup_lr: 1e-6
        min_lr: 1e-5
        lr_noise: Null
        lr_noise_pct: 0.67
        lr_noise_std: 1.0
        seed: 21
        lr_cycle_mul: 1
        lr_cycle_limit: 1

    model_config:
        target: hgvt.network.hypergraph.HypergraphNet
        params:
            in_features: 3
            image_size: 32
            patch_size: 4
            num_classes: 100
            num_layers: 10
            hidden_dim: 64
            num_heads: 2
            adj_dim: Null
            hnode_capacity: 5
            edge_capacity: 8
            hedge_capacity: 4
            split_mode: True
            joint_mlp: True
            bind_qkv: False
            use_stem: True
            dropout: 0.1
            drop_path: 0.1
            drop_decay: False
            skip_last: True
            weight_init: True
            virtual_init: False
            train_pe: False
            use_virtual_pemb: True
            use_diversity_loss: True
            output_ln: True
            use_xformers: False
            pop_level: [10.66, 0.5, 0.5]
            pool_type: 'expert'
            expert_topk: 1
            expert_ce_weight: 0.1
            expert_noise_level: 0.1
            expert_dropout: 0.1
            expert_label_smoothing: 0.0
            expert_use_gate: False

data:
  target: hgvt.data.prefetch_loader.PrefetchLoaderFromConfig
  params:
    use_async: False
    batch_size: 512
    async_config:
        train:
            fp16: True
            re_mode: 'pixel'
            re_prob: 0.25
            re_count: 1
            re_num_splits: 0
            label_smoothing: 0.1
            mixup: 0.8
            cutmix: 1.0
            cutmix_minmax: Null
            mixup_prob: 0.8
            mixup_switch_prob: 0.5
            mixup_mode: 'batch'
            mixup_off_epoch: 0
            num_classes: 100
        val:
            fp16: True
        test:
            fp16: True

    factory_config:
        target: hgvt.data.torchvision_data_module.TorchDataModule
        params:
            batch_size: 512
            dataset_name: CIFAR100
            tar_base: "/vault/imagenet/shards/"
            num_workers: 4
            multinode: False
            pin_memory: True
            generate_cls: True
            train:
              shards: 'train/{00000..00127}.tar'
              shuffle: 10000
              image_key: jpg
              repeat: 1
              repeat_aug: 2
              subset_size: 30000
              cls_dict_path: '/vault/imagenet/imagenet_classdict.json'
              transforms:
                  img_size: 32
                  color_jitter: 0.4
                  re_mode: 'pixel'
                  re_prob: 0.25
                  re_count: 1
                  re_split: False
                  scale: [0.08, 1.0]
                  ratio: [0.75, 1.33]
                  hflip: 0.5
                  vflip: 0.0
                  mean: [0.5, 0.5, 0.5]
                  std: [0.5, 0.5, 0.5]
                  num_aug_splits: 0
                  auto_augment: 'rand-m9-mstd0.5-inc1'
                  interpolation: 'random'
                  use_prefetcher: True
                  crop_pct: Null
            validation:
              shards: 'train/{00128..00128}.tar'
              shuffle: 0
              image_key: jpg
              cls_dict_path: '/vault/imagenet/imagenet_classdict.json'
              repeats: 1
              transforms:
                  mean: [0.5, 0.5, 0.5]
                  std: [0.5, 0.5, 0.5]
                  img_size: 32
                  no_aug: True
                  use_prefetcher: True
            test:
              shards: 'val1/{00000..00007}.tar'
              shuffle: 0
              image_key: jpg
              cls_dict_path: '/vault/imagenet/val_set_id_mapping.json'
              repeats: 1
              subset_size: 5000
              transforms:
                  mean: [0.5, 0.5, 0.5]
                  std: [0.5, 0.5, 0.5]
                  img_size: 32
                  no_aug: True
                  use_prefetcher: True

lightning:
  modelcheckpoint:
    params:
      every_n_train_steps: 500
      save_last: True

  trainer:
    accelerator: gpu
    strategy: ddp_find_unused_parameters_false
    num_sanity_val_steps: 0
    check_val_every_n_epoch: 1
    max_steps: 11800   # a fresh 100 epochs — global_step starts at 0 since we're NOT using --resume
    accumulate_grad_batches: 1
    gradient_clip_val: 1.0

Writing /kaggle/working/hgvt/configs/hgvt_mu_phase2.yaml


In [ ]:
!cd /kaggle/working/hgvt && python train.py \
    --config configs/hgvt_mu_phase2.yaml \
    --name cifar_subset_phase2 \
    --logdir /kaggle/working/logs \
    --gpus 1 \
    --precision bf16

In [ ]:
!zip -r /kaggle/working/working2.zip /kaggle/working

In [16]:
%%writefile /kaggle/working/hgvt/test.py
"""
test.py — one-shot evaluation on the held-out test split.

Deliberately separate from train.py: loads weights only (via
model.params.ckpt_path, same mechanism as the phase-2 config), builds a
bare trainer, and calls trainer.test() exactly once. No resume, no
optimizer, no scheduler — this script has nothing to do with continuing
training.
"""

import argparse
import contextlib
import io

from omegaconf import OmegaConf
import torch
import pytorch_lightning as pl

from hgvt.util import instantiate_from_config


def get_parser():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", type=str, required=True,
        help="Same yaml used for training — reused so the model architecture "
             "and test-data config match exactly.")
    parser.add_argument("--ckpt", type=str, required=True,
        help="Path to the checkpoint to evaluate (e.g. last.ckpt from the "
             "run you want to score). Loaded as weights-only, same as "
             "model.params.ckpt_path — no training state is touched.")
    parser.add_argument("--gpus", type=int, default=1)
    parser.add_argument("--precision", type=str, default="bf16")
    parser.add_argument("--seed", type=int, default=21)
    return parser


def main():
    parser = get_parser()
    args = parser.parse_args()

    pl.seed_everything(args.seed)

    config = OmegaConf.load(args.config)
    model_config = config.model
    data_config = config.data

    # override ckpt_path so this script scores whichever checkpoint you
    # point it at, regardless of what's baked into the yaml
    model_config.params.ckpt_path = args.ckpt

    print(f"Evaluating checkpoint: {args.ckpt}")
    print(f"Using test split from: {args.config}")
    print("Reminder: only score a given model on this split once you're "
          "done tuning it — re-running after changes defeats the point "
          "of a held-out set.\n")

    with contextlib.redirect_stdout(io.StringIO()):
        model = instantiate_from_config(model_config)

    datamodule = instantiate_from_config(data_config)
    inner_factory = getattr(datamodule, "dataset_factory", None)
    if inner_factory is not None and hasattr(inner_factory, "prepare_data"):
        inner_factory.prepare_data()

    use_gpu = args.gpus > 0 and torch.cuda.is_available()
    trainer = pl.Trainer(
        accelerator="gpu" if use_gpu else "cpu",
        devices=args.gpus if use_gpu else 1,
        precision=args.precision,
        logger=False,       # no TB run for a single eval — just read stdout
        enable_checkpointing=False,
    )

    results = trainer.test(model, datamodule=datamodule)

    print("\n=== Test results ===")
    for k, v in results[0].items():
        print(f"{k}: {v:.4f}")


if __name__ == "__main__":
    main()

Writing /kaggle/working/hgvt/test.py


In [17]:
!cd /kaggle/working/hgvt && python test.py \
    --config configs/hgvt_mu_phase2.yaml \
    --ckpt /kaggle/input/datasets/sharifnbd/cifarckpt/last.ckpt \
    --gpus 1 \
    --precision bf16

Seed set to 21
Evaluating checkpoint: /kaggle/input/datasets/sharifnbd/cifarckpt/last.ckpt
Using test split from: configs/hgvt_mu_phase2.yaml
Reminder: only score a given model on this split once you're done tuning it — re-running after changes defeats the point of a held-out set.

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
100%|████████████████████████████████████████| 169M/169M [36:35<00:00, 77.0kB/s]
/usr/local/lib/python3.12/dist-packages/lightning_fabric/connector.py:571: `precision=bf16` is supported for historical reasons but its usage is discouraged. Please set your precision to bf16-mixed instead!
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud l

In [ ]:
%%writefile /kaggle/working/hgvt/eval.py
"""
eval.py — CLI entry point for evaluating a saved HgVT checkpoint.

Not part of the original repo release; mirrors train.py's instantiation
pattern but runs trainer.test() against a checkpoint instead of
trainer.fit(). Reuses the same model/data config as training, so the
model architecture always matches the checkpoint's weights.

Usage:
    python eval.py \
        --config configs/hgvt_mu_smoketest.yaml \
        --ckpt logs/20260713-.../checkpoints/epoch=X-step=Y.ckpt \
        --gpus 1 \
        --precision bf16

Notes:
  - If --ckpt is omitted, evaluates the randomly-initialized model. This
    only exists to smoke-test the eval pipeline itself — the resulting
    accuracy numbers are meaningless.
  - Requires the same distributed-guard patch applied to
    hgvt/hypergraph.py's stop_validation() that was needed for training
    validation to run on a single GPU (test_step routes through the same
    method). If you haven't applied it yet, this will fail the same way
    validation did.
"""

import argparse
import os

from omegaconf import OmegaConf
import torch
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger

from hgvt.util import instantiate_from_config


def get_parser():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--config", type=str, required=True,
        help="Path to the config yaml used for training (architecture must match the checkpoint).",
    )
    parser.add_argument(
        "--ckpt", type=str, default=None,
        help="Path to a .ckpt file to evaluate. If omitted, evaluates random init (pipeline check only).",
    )
    parser.add_argument(
        "--gpus", type=int, default=1,
        help="Number of GPUs to use. 0 = CPU.",
    )
    parser.add_argument(
        "--precision", type=str, default="32",
        help="Eval precision: 32, 16, or bf16.",
    )
    parser.add_argument(
        "--logdir", type=str, default="eval_logs",
        help="Root directory for eval logs.",
    )
    parser.add_argument(
        "--name", type=str, default=None,
        help="Name for this eval run.",
    )
    parser.add_argument(
        "--limit_test_batches", type=str, default=None,
        help="Limit test batches, int or float fraction. Useful for a quick check before a full pass.",
    )
    parser.add_argument(
        "--seed", type=int, default=21,
    )
    return parser


def parse_int_or_float(value):
    if value is None:
        return None
    try:
        return int(value)
    except ValueError:
        return float(value)


def build_trainer(args, logger, limit_test_batches):
    use_gpu = args.gpus > 0 and torch.cuda.is_available()
    if args.gpus > 0 and not torch.cuda.is_available():
        print("WARNING: --gpus > 0 requested but no CUDA device is visible. Falling back to CPU.")

    accelerator = "gpu" if use_gpu else "cpu"
    devices = args.gpus if use_gpu else 1

    kwargs = {}
    if limit_test_batches is not None:
        kwargs["limit_test_batches"] = limit_test_batches

    try:
        trainer = pl.Trainer(
            accelerator=accelerator,
            devices=devices,
            precision=args.precision,
            logger=logger,
            **kwargs,
        )
    except Exception as e:
        if args.precision in ("16", "bf16"):
            print(f"Trainer construction failed with precision='{args.precision}' ({e}); "
                  f"retrying with '{args.precision}-mixed'.")
            trainer = pl.Trainer(
                accelerator=accelerator,
                devices=devices,
                precision=f"{args.precision}-mixed",
                logger=logger,
                **kwargs,
            )
        else:
            raise
    return trainer


def main():
    parser = get_parser()
    args = parser.parse_args()

    pl.seed_everything(args.seed)

    config = OmegaConf.load(args.config)
    model_config = config.model
    data_config = config.data

    run_name = args.name or (os.path.splitext(os.path.basename(args.config))[0] + "_eval")

    print(f"Config: {args.config}")
    print(f"Checkpoint: {args.ckpt if args.ckpt else '(none — random init, pipeline check only)'}")

    # --- build model ---
    model = instantiate_from_config(model_config)

    # --- build data module ---
    datamodule = instantiate_from_config(data_config)

    # same fix as train.py: PrefetchLoaderFromConfig wraps an inner
    # factory (e.g. TorchDataModule) whose own prepare_data() (CIFAR
    # download) PL never calls automatically.
    inner_factory = getattr(datamodule, "dataset_factory", None)
    if inner_factory is not None and hasattr(inner_factory, "prepare_data"):
        print("Preparing/downloading dataset via inner data factory...")
        inner_factory.prepare_data()

    # --- logger ---
    logger = TensorBoardLogger(save_dir=args.logdir, name=run_name)

    # --- trainer ---
    limit_test_batches = parse_int_or_float(args.limit_test_batches)
    trainer = build_trainer(args, logger, limit_test_batches)

    # --- go ---
    # ckpt_path=None evaluates the model's current (randomly-initialized)
    # in-memory weights; a real path loads weights from that checkpoint
    # before testing.
    results = trainer.test(model, datamodule=datamodule, ckpt_path=args.ckpt)
    print("Test results:")
    print(results)


if __name__ == "__main__":
    main()

In [ ]:
!find /kaggle/working/logs -name "*.ckpt" -exec ls -lh {} \;

In [ ]:
path = "/kaggle/working/hgvt/hgvt/hypergraph.py"
with open(path) as f:
    content = f.read()

old = "    def test_epoch_end(self, *args, **kwargs):"
new = "    def on_test_epoch_end(self, *args, **kwargs):"

assert old in content, "pattern not found — check the file matches expected content"
content = content.replace(old, new, 1)

with open(path, "w") as f:
    f.write(content)

print("patched: test_epoch_end -> on_test_epoch_end")

In [ ]:
!cd /kaggle/working/hgvt && python eval.py \
    --config configs/hgvt_mu_smoketest.yaml \
    --ckpt "/kaggle/working/logs/20260713-100057_val_epoch_check2/checkpoints/epoch=935-step=4680.ckpt" \
    --gpus 1 \
    --precision bf16